<a href="https://colab.research.google.com/github/vedantyeole0207/Political-Media-Bias-Prediction/blob/vedantyeole0207%2FUsed-Car-Price-Prediction-Full-Stack/Political_Media_Bias.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [136]:
import os

os.environ["TF_XLA_FLAGS"] = "--tf_xla_enable_xla_devices=false"

os.environ["TF_DISABLE_JIT"] = "1"

In [137]:
print("TF version:", tf.__version__)
print("JIT enabled (should be False):", tf.config.optimizer.get_jit())
print("GPUs:", tf.config.list_physical_devices("GPU"))

TF version: 2.19.1
JIT enabled (should be False): 
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


**IMPORTING SOME NECESSARY LIBRARIES**

In [138]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
import re
import nltk
from nltk.corpus import stopwords
import numpy as np
from nltk.stem import WordNetLemmatizer
from sklearn.compose import ColumnTransformer
import tensorflow as tf
import keras_nlp
import pandas as pd
from sklearn.model_selection import train_test_split

**IMPORTING DATASET**

In [139]:
file_path = 'political_social_media.csv'
data = pd.read_csv(file_path, encoding='latin1')

In [140]:
data.head(5)

,_unit_id,_golden,_unit_state,_trusted_judgments,_last_judgment_at,audience,audience:confidence,bias,bias:confidence,message,...,orig__golden,audience_gold,bias_gold,bioid,embed,id,label,message_gold,source,text
0,766192484,False,finalized,1,8/4/15 21:17,national,1.0,partisan,1.0,policy,...,NaN,NaN,NaN,R000596,"<blockquote class=""twitter-tweet"" width=""450"">...",3.83249E+17,From: Trey Radel (Representative from Florida),NaN,twitter,RT @nowthisnews: Rep. Trey Radel (R- #FL) slam...
1,766192485,False,finalized,1,8/4/15 21:20,national,1.0,partisan,1.0,attack,...,NaN,NaN,NaN,M000355,"<blockquote class=""twitter-tweet"" width=""450"">...",3.11208E+17,From: Mitch McConnell (Senator from Kentucky),NaN,twitter,VIDEO - #Obamacare: Full of Higher Costs and ...
2,766192486,False,finalized,1,8/4/15 21:14,national,1.0,neutral,1.0,support,...,NaN,NaN,NaN,S001180,"<blockquote class=""twitter-tweet"" width=""450"">...",3.39069E+17,From: Kurt Schrader (Representative from Oregon),NaN,twitter,Please join me today in remembering our fallen...
3,766192487,False,finalized,1,8/4/15 21:08,national,1.0,neutral,1.0,policy,...,NaN,NaN,NaN,C000880,"<blockquote class=""twitter-tweet"" width=""450"">...",2.98528E+17,From: Michael Crapo (Senator from Idaho),NaN,twitter,RT @SenatorLeahy: 1st step toward Senate debat...
4,766192488,False,finalized,1,8/4/15 21:26,national,1.0,partisan,1.0,policy,...,NaN,NaN,NaN,U000038,"<blockquote class=""twitter-tweet"" width=""450"">...",4.07643E+17,From: Mark Udall (Senator from Colorado),NaN,twitter,.@amazon delivery #drones show need to update ...


**KEEPING RELEVANT COLUMNS**

In [141]:
data = data[['text', 'bias']].dropna()

# Inspect label distribution
print(data['bias'].value_counts())

bias
neutral     3689
partisan    1311
Name: count, dtype: int64


**DATA CLEANING**

In [142]:
import re
def clean_text(text):
    text = re.sub(r"http\S+|www\S+|https\S+", '', text, flags=re.MULTILINE)
    text = re.sub(r'\@\w+|\#','', text)
    return text.strip()

data['clean_text'] = data['text'].apply(clean_text)
data.head()


,text,bias,clean_text
0,RT @nowthisnews: Rep. Trey Radel (R- #FL) slam...,partisan,RT : Rep. Trey Radel (R- FL) slams Obamacare. ...
1,VIDEO - #Obamacare: Full of Higher Costs and ...,partisan,VIDEO - Obamacare: Full of Higher Costs and B...
2,Please join me today in remembering our fallen...,neutral,Please join me today in remembering our fallen...
3,RT @SenatorLeahy: 1st step toward Senate debat...,neutral,RT : 1st step toward Senate debate on Leahy-Cr...
4,.@amazon delivery #drones show need to update ...,partisan,. delivery drones show need to update law to p...


In [143]:
data['label'] = data['bias'].map({'neutral': 0, 'partisan': 1})

In [144]:
from sklearn.utils import resample

majority_class = data[data.label == 0]
minority_class = data[data.label == 1]

minority_oversampled = resample(minority_class,
                                 replace=True,
                                 n_samples=len(majority_class),
                                 random_state=42)

data = pd.concat([majority_class, minority_oversampled])

In [146]:
print(data['label'].value_counts())

label
0    3689
1    3689
Name: count, dtype: int64


**MODEL TRAINING USING TF-IDF VECTORIZER**

In [147]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

X_train, X_test, y_train, y_test = train_test_split(
    data['clean_text'], data['label'], test_size=0.2, random_state=42, stratify=data['bias'])

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2), stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [148]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

model = LogisticRegression(max_iter=300)
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.8285907859078591
              precision    recall  f1-score   support

           0       0.85      0.79      0.82       738
           1       0.81      0.86      0.83       738

    accuracy                           0.83      1476
   macro avg       0.83      0.83      0.83      1476
weighted avg       0.83      0.83      0.83      1476



In [150]:
sample = ["The administration is destroying our country!",
          "Today marks a great milestone in healthcare reform."]
sample_tfidf = vectorizer.transform(sample)
print(model.predict(sample_tfidf))

[1 0]


**AS WE KNOW TF-IDF IS UNABLE TO CAPTURE THE CONTEXT OF SENTENCE, WE WILL NOW TRY BERT MODEL** **bold text**

In [151]:
train_texts = X_train.tolist()
test_texts = X_test.tolist()
train_labels = y_train.astype(int).tolist()
test_labels = y_test.astype(int).tolist()

train_ds = tf.data.Dataset.from_tensor_slices((train_texts, train_labels)).shuffle(1000).batch(16)
test_ds = tf.data.Dataset.from_tensor_slices((test_texts, test_labels)).batch(16)


In [152]:
model2 = keras_nlp.models.BertClassifier.from_preset("bert_base_en_uncased", num_classes=2)

In [153]:
model2.compile(
    optimizer=tf.keras.optimizers.Adam(3e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

In [154]:
history = model2.fit(train_ds, validation_data=test_ds, epochs=3)

Epoch 1/3
369/369 ━━━━━━━━━━━━━━━━━━━━ 699s 2s/step - accuracy: 0.7256 - loss: 0.5615 - val_accuracy: 0.8388 - val_loss: 0.4006
Epoch 2/3
369/369 ━━━━━━━━━━━━━━━━━━━━ 578s 2s/step - accuracy: 0.8713 - loss: 0.3254 - val_accuracy: 0.8963 - val_loss: 0.2744
Epoch 3/3
369/369 ━━━━━━━━━━━━━━━━━━━━ 578s 2s/step - accuracy: 0.9607 - loss: 0.1253 - val_accuracy: 0.9045 - val_loss: 0.3141


In [155]:
model2.evaluate(test_ds)

93/93 ━━━━━━━━━━━━━━━━━━━━ 47s 499ms/step - accuracy: 0.9043 - loss: 0.3135


[0.3140682280063629, 0.9044715166091919]

**PREDICTING DATA FOR UNSEEN ARTICLES**

In [162]:
samples = [
    # 1. Expected: partisan (negative opinion)
    "The proposed tax hike is a disastrous move that will crush small businesses and cripple the economy.",

    # 2. Expected: neutral (factual report)
    "The bill requires companies with over 50 employees to offer healthcare benefits.",

    # 3. Expected: partisan (positive spin)
    "This brilliant and compassionate legislation is a landmark victory for working families across the nation.",

    # 4. Expected: neutral (diplomatic update)
    "Diplomats from both countries met for three hours to discuss the ongoing border dispute.",

    # 5. Expected: partisan (fear-based language)
    "Failing to pass this bill will leave our nation vulnerable and endanger our security.",

    # 6. Expected: neutral (statistical report)
    "According to the Department of Commerce, retail sales increased by 0.5% in the last quarter.",

    # 7. Expected: partisan (unverified claim)
    "It's clear the opposition's entire platform is built on nothing but lies and misinformation."
]

In [163]:
preds = model2.predict(samples)

1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step


In [164]:
preds

array([[-2.983235  ,  3.6403081 ],
       [-0.77725947,  1.2009062 ],
       [ 1.2039644 , -1.3806154 ],
       [ 1.3135555 , -1.2701787 ],
       [-2.195831  ,  3.1082025 ],
       [ 2.9578805 , -2.9709642 ],
       [-2.2372682 ,  2.8473032 ]], dtype=float32)

In [165]:
labels = tf.argmax(preds, axis=1).numpy()

In [166]:
labels

array([1, 1, 0, 0, 1, 0, 1])

In [167]:
for text, label in zip(samples, labels):
    print(f"{text} -> {'partisan' if label==1 else 'neutral'}")

The proposed tax hike is a disastrous move that will crush small businesses and cripple the economy. -> partisan
The bill requires companies with over 50 employees to offer healthcare benefits. -> partisan
This brilliant and compassionate legislation is a landmark victory for working families across the nation. -> neutral
Diplomats from both countries met for three hours to discuss the ongoing border dispute. -> neutral
Failing to pass this bill will leave our nation vulnerable and endanger our security. -> partisan
According to the Department of Commerce, retail sales increased by 0.5% in the last quarter. -> neutral
It's clear the opposition's entire platform is built on nothing but lies and misinformation. -> partisan


**THEREFORE WE CAN CONCLUDE THAT BERT MODEL IS BETTER THAN TF-IDF IN THIS PROBLEM STATEMENT**